# 🧠 NOVA: Autonomous Reasoning Training

Welcome to the training environment! This notebook uses the **Unsloth** library to fine-tune a Llama-3 model.

### 🚨 CRITICAL: Enable GPU First!
Unsloth requires a GPU to run. If you see an error about `Torch not compiled with CUDA`, do this:
1. Click **Runtime** in the top menu.
2. Select **Change runtime type**.
3. Set **Hardware accelerator** to **T4 GPU**.
4. Click **Save** and then run the cells again.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise Exception("❌ GPU NOT DETECTED! Please go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU' before continuing.")

print("✅ GPU Detected: ", torch.cuda.get_device_name(0))

major_version, minor_version = torch.cuda.get_device_capability()
print(f"CUDA Capability: {major_version}.{minor_version}")

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
pass

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    lof_weight = 0,
)

### 📁 Upload Dataset
Upload your `nova_skills_dataset.jsonl` file.

In [ ]:
from google.colab import files
import json
from datasets import Dataset

uploaded = files.upload()
filename = list(uploaded.keys())[0]

data = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

def format_prompts(examples):
    instructions = []
    for messages in examples["messages"]:
        user_msg = messages[0]["content"]
        assistant_msg = messages[1]["content"]
        prompt = f"USER_INPUT: {user_msg}\n\nRESPONSE: {assistant_msg}"
        instructions.append(prompt)
    return { "text" : instructions }

dataset = Dataset.from_list([{"messages": d["messages"]} for d in data])
dataset = dataset.map(format_prompts, batched = True)
print(f"Loaded {len(dataset)} examples successfully!")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
print("💾 Saving LoRA Adapter...")
model.save_pretrained("nova_skill_adapter")
tokenizer.save_pretrained("nova_skill_adapter")

import shutil
shutil.make_archive("nova_skill_adapter", 'zip', "nova_skill_adapter")
print("✅ All done! Downloading zip...")
files.download("nova_skill_adapter.zip")